In [35]:
import pandas as pd
import os
import numpy as np
from datetime import datetime
import ast
from googletrans import Translator
from concurrent.futures import ThreadPoolExecutor, as_completed
import asyncio, random
from concurrent.futures import ProcessPoolExecutor
from concurrent.futures import ThreadPoolExecutor
from transformers import CLIPTokenizer, CLIPTextModel
from openai import OpenAI
import json
import re
import anthropic
import torch
from datetime import datetime
from sklearn.metrics.pairwise import cosine_similarity
from google import genai
from google.genai import types

In [2]:
artnet_2025= pd.read_excel("D:\\MissTiny\\GitHub\\Creativity_Artnet\\Datasets\\ArtNet\\Artnet_Europe.xlsx")

In [3]:
artnet_2025.columns

Index(['ID', 'lot id', 'artwork id', 'artist id', 'sale date',
       'artist modifier', 'first', 'last', 'nationality', 'year born',
       'year died', 'title', 'workyear modifier', 'workyear from',
       'workyear to', 'est lo', 'est hi', 'sale price', 'currency',
       'currency exchange rate', 'est lo usd', 'est hi usd', 'sale price usd',
       'priceStatus', 'pricePhrase', 'has_image', 'sale year', 'artist'],
      dtype='object')

# Test on Consistency

In [16]:
artnet_2024_Picasso = pd.read_excel("D:\\MissTiny\\GitHub\\Creativity_Artnet\\Datasets\\ArtNet\\Artnet_Europe_2024_Picasso.xlsx")

In [36]:
with open("D:\\MissTiny\\GitHub\\Creativity_Chess\\Token_Key\\gemini_api.txt", "r", encoding="utf-8") as file:
    gemini_api = file.read()

In [37]:
gemini_client = genai.Client(
    api_key=gemini_api
)

In [38]:
def annotation(i,content):
    row=[content["artwork id"],content["title"],content['first'],content['last'],content['workyear from'],content["nationality"]]

    instruction_message=fr"""You are an expert in artwork criticism, art history, and creativity evaluation.
    Using web search, find authoritative commentary on the artistic value and creative significance of the artwork:
    - Title: \'{content["title"]}\' 
    - Artist:{content['first']} {content['last']} ({content["nationality"]})
    - Year of Creation:{content['workyear from']} 
    
    Based strictly on information gathered through web search, summarize the consensus and key scholarly interpretations in English regarding the artwork’s artistic value and creativity.
    Your analysis should address innovations in composition, technique, emotional expression, iconography, and influences on later art.
    - An artwork is creative only if it is new, valuable, and surprising compared to prior artworks
    
    OUTPUT FORMAT(around 150 words each)

    **Artistic Value: <High or Low>**
    <Your summary here>
    
    **Creativity: <Yes or No>**
    <Your summary here>
    """ 
    grounding_tool = types.Tool(
        google_search=types.GoogleSearch()
    )
    
    config = types.GenerateContentConfig(
        tools=[grounding_tool]
    )
    response =gemini_client.models.generate_content(
        model="gemini-2.5-flash-lite",
        contents=instruction_message,
        config=config,
    )
    answer = response.text
    artistic_answer = re.search(r"\*\*Artistic Value:\s*([^*]+)\*\*", answer)
    artistic_comment = re.search(r"\*\*Artistic Value:[^*]+\*\*\s*(.*?)\n\n\*\*Creativity", answer, re.S)

    creativity_answer = re.search(r"\*\*Creativity:\s*([^*]+)\*\*", answer)
    creativity_comment = re.search(r"\*\*Creativity:[^*]+\*\*\s*(.*)", answer, re.S)

    if not (artistic_answer and artistic_comment and creativity_answer and creativity_comment):
        row=[content["artwork id"], content["title"], content['first'], content['last'], content['workyear from'], content["nationality"],"","","",""]
        return i, row, answer
    
    result = {
        "artistic_value_answer": artistic_answer.group(1).strip() if artistic_answer else "",
        "artistic_value_comment": artistic_comment.group(1).strip() if artistic_comment else "",
        "creativity_answer": creativity_answer.group(1).strip() if creativity_answer else "",
        "creativity_comment": creativity_comment.group(1).strip() if creativity_comment else ""
    }

    row.append(result['artistic_value_answer'])
    row.append(result['artistic_value_comment'])
    row.append(result['creativity_answer'])
    row.append(result['creativity_comment'])
    return i, row, ""


In [12]:
# number_size= 5000
# number_size=df_raw.shape[0]
# range_start = 30000
range_start =4650
# range_end = min(range_start+number_size,artnet_2024_Picasso.shape[0])
range_end = 5000
number_size = range_end - range_start
N = min(number_size, artnet_2024_Picasso.shape[0]-range_start)

In [13]:
error_output=[]
annotation_result = pd.DataFrame({
    "artwork id": np.full(number_size, np.nan, dtype=int),
    "title": [""] * number_size,
    "first": [""] * number_size,
    "last": [""] * number_size,
    "workyear from":np.full(number_size, np.nan, dtype=int),
    "nationality":[""] * number_size,
    "artistic_value_answer": [""] * number_size,
    "artistic_value_comment":[""] * number_size,
    "creativity_answer": [""] * number_size,
    "creativity_comment": [""] * number_size})
max_workers = 20  # tune to your CPU cores
print(f"{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}: Parallel computation starts")
with ThreadPoolExecutor(max_workers=max_workers) as ex:
    futures = {
        ex.submit(annotation, i,
                   {
                "artwork id": artnet_2024_Picasso.iloc[i]["artwork id"],
                "title": artnet_2024_Picasso.iloc[i]["title"],
                "first": artnet_2024_Picasso.iloc[i]["first"],
                "last": artnet_2024_Picasso.iloc[i]["last"],
                "workyear from": artnet_2024_Picasso.iloc[i]["workyear from"],
                "nationality": artnet_2024_Picasso.iloc[i]["nationality"]}
                ): i
        for i in range(range_start,range_end)
    }
    
    completed = 0
    for fut in as_completed(futures):
        i,result,error = fut.result()
        annotation_result.loc[i-range_start]  = result
        if error !="":
            error_output.append(error)
        completed += 1
        if completed % 50 == 0:
            print(f"{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}: Completed {completed}/{len(artnet_2024_Picasso)}")
print(f"{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}: Ends")
print(f"{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}: There are {len(error_output)} errors")

C:\Users\MissTiny\anaconda3\envs\Creativity\Lib\site-packages\numpy\_core\numeric.py:353: RuntimeWarning: invalid value encountered in cast
  multiarray.copyto(a, fill_value, casting='unsafe')


2025-12-04 05:29:51: Parallel computation starts
2025-12-04 05:30:06: Completed 50/40558
2025-12-04 05:30:18: Completed 100/40558
2025-12-04 05:30:31: Completed 150/40558
2025-12-04 05:30:43: Completed 200/40558
2025-12-04 05:30:56: Completed 250/40558
2025-12-04 05:31:08: Completed 300/40558
2025-12-04 05:31:20: Completed 350/40558
2025-12-04 05:31:20: Ends
2025-12-04 05:31:20: There are 0 errors


In [14]:
annotation_result.to_excel("comment_annotation_gemini_5000.xlsx",index=False)

# Check and Fix

In [38]:
empty_idx = annotation_result.index[annotation_result["artistic_value_answer"] == ""].tolist()

In [39]:
print(f"{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}: There are {len(empty_idx)} errors")

2025-11-28 07:48:10: There are 1 errors


In [40]:
print(f"{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}: Start")
count= 0
for error_index in empty_idx:
    if count%10 ==0:
        print(f"{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}: Current at {count}")
    content = {
                "artwork id": artnet_2024_Picasso.iloc[error_index]["artwork id"],
                "title": artnet_2024_Picasso.iloc[error_index]["title"],
                "first": artnet_2024_Picasso.iloc[error_index]["first"],
                "last": artnet_2024_Picasso.iloc[error_index]["last"],
                "workyear from": artnet_2024_Picasso.iloc[error_index]["workyear from"],
                "nationality": artnet_2024_Picasso.iloc[error_index]["nationality"]}
    _,result,error = annotation(error_index,content)
    annotation_result.loc[error_index,:] =result
    count+=1
    # if count ==10:
    #     break
print(f"{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}: Ends")

2025-11-28 07:48:16: Start
2025-11-28 07:48:16: Current at 0
2025-11-28 07:48:20: Ends


In [41]:
annotation_result.index[annotation_result["artistic_value_answer"] == ""].tolist()

[]

In [42]:
print(f"{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}: There are {len(annotation_result.index[annotation_result["artistic_value_answer"] == ""].tolist())} errors")

2025-11-28 07:48:23: There are 0 errors


# Error Fix

In [25]:
artnet_2024_Picasso = pd.read_excel("D:\\MissTiny\\GitHub\\Creativity_Artnet\\Datasets\\ArtNet\\Artnet_Europe_2024_Picasso.xlsx")

In [26]:
annotation_result = pd.read_excel("comment_annotation_gemini_final.xlsx")

In [27]:
sum(annotation_result['artwork id'].duplicated())

0

In [74]:
check_point = annotation_result['artwork id'] == artnet_2024_Picasso.iloc[0:5000]['artwork id']

In [78]:
a = artnet_2024_Picasso.iloc[0:5000][annotation_result['artwork id'] != artnet_2024_Picasso.iloc[0:5000]['artwork id']]

In [79]:
a.shape

(0, 16)

In [77]:
print(f"{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}: Start")
count=0
for i in range(5000):
    if not check_point[i]:
        content = {
            "artwork id": artnet_2024_Picasso.iloc[i]["artwork id"],
            "title": artnet_2024_Picasso.iloc[i]["title"],
            "first": artnet_2024_Picasso.iloc[i]["first"],
            "last": artnet_2024_Picasso.iloc[i]["last"],
            "workyear from": artnet_2024_Picasso.iloc[i]["workyear from"],
            "nationality": artnet_2024_Picasso.iloc[i]["nationality"]}
        _,result,error = annotation(i,content)
        annotation_result.loc[i,:] =result
        count+=1
        if count%50 ==0:
            print(f"{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}: Current at {count}")
print(f"{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}: Ends")

2025-12-04 15:52:24: Start
2025-12-04 15:56:09: Current at 50
2025-12-04 15:59:33: Ends


In [80]:
annotation_result.to_excel("comment_annotation_gemini_fix.xlsx",index=False)

# Test on Prompt Sensitivity Code

In [5]:
with open("D:\\MissTiny\\GitHub\\Creativity_Chess\\Token_Key\\gemini_api.txt", "r", encoding="utf-8") as file:
    gemini_api = file.read()

In [6]:
row = artnet_2025.iloc[1]

In [7]:
instruction_message=f"""You are an expert in artwork criticism, art history, and creativity evaluation. 
Using web search, find authoritative commentary on the artistic value and creative significance of the artwork:
- Title: \'{row.title}\' 
- Artist:{row['first']} {row['last']} ({row.nationality})
- Year of Creation:{row['workyear from']} 

Based strictly on information gathered through web search, summarize the consensus and key scholarly interpretations regarding the artwork’s artistic value and creativity.
Your analysis should address innovations in composition, technique, emotional expression, iconography, and influences on later art.
- An artwork is creative only if it is new, valuable, and surprising compared to prior artworks

OUTPUT FORMAT(around 150 words each)
Artistic Value:
<Your summary here>
Creativity
<Your summary here>
"""

In [8]:
print(instruction_message)

You are an expert in artwork criticism, art history, and creativity evaluation. 
Using web search, find authoritative commentary on the artistic value and creative significance of the artwork:
- Title: 'La Vierge d'humilité, tableau de dévotion' 
- Artist:Francesco di Gentile da Fabriano (Italian)
- Year of Creation:1420 

Based strictly on information gathered through web search, summarize the consensus and key scholarly interpretations regarding the artwork’s artistic value and creativity.
Your analysis should address innovations in composition, technique, emotional expression, iconography, and influences on later art.
- An artwork is creative only if it is new, valuable, and surprising compared to prior artworks

OUTPUT FORMAT(around 150 words each)
Artistic Value:
<Your summary here>
Creativity
<Your summary here>



In [13]:
gemini_client = genai.Client(
    api_key=gemini_api
)

In [15]:
response =gemini_client.models.generate_content(
    model="gemini-2.5-flash-lite",
    contents=instruction_message,
    config={
            "tools": [
                {
                    "google_search": {}
                }
            ]
        }
)

In [17]:
print(response.text)

**Artistic Value:**

Francesco di Gentile da Fabriano's "La Vierge d'humilité, tableau de dévotion" (c. 1420) is a significant work that bridges the International Gothic style with emerging Renaissance sensibilities. Artistically, its value lies in its delicate linearity, rich decorative elements, and vibrant colors, characteristic of the International Gothic style. The painting showcases Gentile's mastery in detailed representations, drawing from his observations of the natural world. The iconography of the Madonna of Humility itself, depicting Mary seated on the ground, emphasizes her humanity and approachability, a departure from the more formal, enthroned Madonnas. This particular work, though potentially intended for private devotion due to its scale, demonstrates a refinement and sophistication that speaks to the era's developing artistic tastes. The use of tempera and gold leaf, common for devotional pieces of the period, contributes to its preciousness and spiritual aura. While

In [18]:
prompt1=f"""You are an expert in artwork criticism, art history, and creativity evaluation.
Using web search, find authoritative commentary on the artistic value and creative significance of the artwork:
- Title: \'{row.title}\' 
- Artist:{row['first']} {row['last']} ({row.nationality})
- Year of Creation:{row['workyear from']} 

Based strictly on information gathered through web search, summarize the consensus and key scholarly interpretations in English regarding the artwork’s artistic value and creativity.
Your analysis should address innovations in composition, technique, emotional expression, iconography, and influences on later art.
- An artwork is creative only if it is new, valuable, and surprising compared to prior artworks

OUTPUT FORMAT(around 150 words each)
Artistic Value: 
<Your summary here>
Creativity: <Yes or No>
<Your summary here>
"""

In [19]:
response1 =gemini_client.models.generate_content(
    model="gemini-2.5-flash-lite",
    contents=prompt1,
    config={
            "tools": [
                {
                    "google_search": {}
                }
            ]
        }
)

In [20]:
print(response1.text)

The artwork 'La Vierge d'humilité, tableau de dévotion' by Francesco di Gentile da Fabriano, created around 1420, is highly regarded for its artistic value, particularly within the International Gothic style. Scholars note Gentile da Fabriano's mastery in combining precious materials with a keen interest in naturalism, using real gold to create subtle lighting effects that lend the figures a lifelike quality. The use of light and the depiction of three-dimensional figures contribute to defining space and bringing the artwork to life. His detailed rendering of faces with varying expressions was also innovative for the time. The painting's decorative richness, evident in the luxurious fabrics and the inclusion of exotic elements and animals, speaks to the patron's wealth and status while contributing to an exotic setting for the biblical scene. The inclusion of Arabic imitation script further enhances its exoticism and luxury. These elements, combined with fine linearity and vibrant colo

In [21]:
prompt2=f"""Act as an expert in artwork criticism, art history, and creativity evaluation.
Using web search, summarize in English the artistic value and creative significance of {row['first']} {row['last']}’s {row['workyear from']}  artwork “{row.title}”.
Note that an artwork is creative only if it is new, valuable, and surprising compared to prior artworks

OUTPUT: (~150 words each)
Artistic Value:
<text>

Creativity: <Yes or No>
<text>
"""

In [22]:
response2 =gemini_client.models.generate_content(
    model="gemini-2.5-flash-lite",
    contents=prompt2,
    config={
            "tools": [
                {
                    "google_search": {}
                }
            ]
        }
)

In [23]:
print(response2.text)

Artistic Value:
Francesco di Gentile da Fabriano's "La Vierge d'humilité, tableau de dévotion" (1420) is a significant work within the context of early 15th-century Italian painting. Its value lies in its rich execution and devotional intensity. The painting likely exhibits the detailed and decorative style characteristic of the International Gothic, a style that was prevalent across Europe. This would involve meticulous rendering of fabrics, intricate patterns, and a jewel-like quality in the colors, creating a luxurious and aesthetically pleasing surface. The "humble Virgin" theme, a less monumental depiction of Mary and Child compared to previous portrayals, offers a more intimate and accessible devotional experience for the viewer. This focus on humanizing the divine, presenting a more tender and relatable Mary, contributes to its artistic merit by fostering a closer connection between the faithful and the sacred figures. The use of gold leaf or gilded elements, common in devotiona

In [24]:
prompt3=f"""ou are a general expert in artwork criticism, art history, and creativity evaluation.
Using web search, analyze the artwork:

- Title: '{row.title}'
- Artist: {row['first']} {row['last']} ({row.nationality})
- Year: {row['workyear from']}

Based ONLY on web search results, provide two ~150-word sections evaluating:
1. Artistic Value
2. Creativity

Note that an artwork is creative only if it is new, valuable, and surprising compared to prior artworks

OUTPUT:
Artistic Value:
<text>
Creativity: <Yes or No>
<text>
"""

In [25]:
response3 =gemini_client.models.generate_content(
    model="gemini-2.5-flash-lite",
    contents=prompt3,
    config={
            "tools": [
                {
                    "google_search": {}
                }
            ]
        }
)

In [26]:
print(response3.text)

**Artistic Value:**
Francesco di Gentile da Fabriano's "La Vierge d'humilité, tableau de dévotion" (circa 1420) is a significant work within the context of early Italian Renaissance devotional art. The painting exemplifies the International Gothic style with its elegant refinement and meticulous attention to detail, particularly in the rendering of fabrics and the surrounding flora, which carry symbolic meaning of the Virgin's virtues. While Gentile da Fabriano is recognized as a leading exponent of this style, his work also contributed to the emerging trends that foreshadowed the Renaissance, such as his unifying use of light to suggest dimension and perspective. The work's modest scale suggests it was intended for private devotion, a common practice for small devotional pictures produced in large numbers during this period for both churches and individuals. The use of a golden background, while a nod to older icon traditions, is here supplemented by a Tuscan panorama, indicating a mo

In [27]:
prompt4=f"""Assume the role of a comprehensive art expert with deep knowledge of global art history, visual analysis, and creativity theory.
Using web search, gather authoritative scholarly commentary on:

- '{row.title}' by {row['first']} {row['last']} ({row.nationality}), created in {row['workyear from']}.

Based strictly on verified search results, summarize the artwork’s:
1. Artistic Value — aesthetic qualities, emotional resonance, technical execution, symbolism, and critical reception.
2. Creativity — an artwork is creative only if it is new, valuable, and surprising compared to prior artworks.

Write ~150 words per section, in English.

FORMAT:
Artistic Value:
<text>

Creativity: <Yes or No>
<text>
"""

In [28]:
response4 =gemini_client.models.generate_content(
    model="gemini-2.5-flash-lite",
    contents=prompt4,
    config={
            "tools": [
                {
                    "google_search": {}
                }
            ]
        }
)

In [29]:
print(response4.text)

**Artistic Value:**
"La Vierge d'humilité" by Francesco di Gentile da Fabriano, created around 1420, is a devotional painting that exemplifies the transition from Gothic aesthetics to early Renaissance sensibilities. The artwork is noted for its meticulous detail, particularly in the depiction of flora, which serves a dual purpose: symbolic and naturalistic. The strawberries and violets, for instance, are rendered with a descriptive precision that links to a search for truth while retaining a Gothic decorative spirit. This naturalistic observation aligns with the contemporary interest in the tangible world, moving away from purely abstract representations. The composition itself, featuring the Virgin Mary seated on the ground, emphasizes her humanity and humility, a departure from the more formal and distant depictions of the "Maestà." The refined execution, characteristic of the Sienese and International Gothic styles, is evident in the elegant lines of Mary's mantle and the decorativ

In [30]:
tokenizer = CLIPTokenizer.from_pretrained("openai/clip-vit-base-patch32")
text_model = CLIPTextModel.from_pretrained("openai/clip-vit-base-patch32")

In [31]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = text_model.to(device)

In [34]:
texts=[response1.text,response2.text,response3.text,response4.text]

In [35]:
inputs = tokenizer(texts, padding=True, truncation=True, return_tensors="pt").to(device)
with torch.no_grad():
    outputs = model(**inputs)
text_embeds = outputs.pooler_output
text_embeds = text_embeds / text_embeds.norm(dim=-1, keepdim=True)
text_embeds=text_embeds.cpu().numpy()

In [36]:
pairwise = cosine_similarity(text_embeds)

In [37]:
pairwise

array([[1.0000002 , 0.9281757 , 0.87428695, 0.90575755],
       [0.9281757 , 0.9999999 , 0.9134153 , 0.9188012 ],
       [0.87428695, 0.9134153 , 1.0000005 , 0.9176482 ],
       [0.90575755, 0.9188012 , 0.9176482 , 1.0000001 ]], dtype=float32)